# QA Generation

### Setup & imports

In [16]:
# Setup & imports
from pathlib import Path
import ntpath
from datetime import datetime
import sys
import pandas as pd
import re
import json

In [ ]:
# Setup & imports

import os
import base64
from openai import AzureOpenAI

endpoint = os.getenv("ENDPOINT_URL", "")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", "")  

# Initialize Azure OpenAI client with key-based authentication",
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)

In [18]:

#prompt builder

def build_prompt(passages, system_prompt):

    user_content = []

    for passage in passages:
        user_content.append({"type": "text",
                             "text": passage})

    chat_prompt = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_prompt
                }
            ]
        },
        {
            "role": "user",
            "content": user_content
        }
    ]
    return chat_prompt


In [19]:
# Modeling & evaluation
def openai_api(lines, system_prompt):
    # Include speech result if speech is enabled
    messages = build_prompt(lines, system_prompt)
    
    # Generate the completion
    completion = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=2000,
        temperature=0.7,
        top_p=0.95,
        frequency_penalty=0,
        presence_penalty=0,
        stop=None,
        stream=False
    )
    
    return completion.choices[0].message.content

In [ ]:
# Data loading
existing_QAs = pd.read_csv('./augmented/query_set.csv')

In [ ]:
# Data loading
QAs = []
gen = os.walk("./augmented/conversations")
next(gen)
for x in gen:
    directory = Path(x[0])
    text_files = list(directory.rglob('*.txt'))
    for file in text_files:
        with open(file) as file:
            lines = [line.rstrip() for line in file]
            if ": Image: [" in f"\n".join(lines):
                original_datetime_string = lines[0][6:]
                if existing_QAs["file"].str.contains(original_datetime_string).any(): 
                    response = openai_api(lines, "Process the provided conversation snippet and generate a single question-answer pair that the author might retrospectively ask about the image described in the conversation. Focus on creating questions that are directly related to the image and its context within the conversation. Keep both the question and answer concise and straightforward.\n\n# Steps\n\n1. Analyze the conversation snippet to identify the image described and its details.\n2. Formulate a simple question that the author might ask after reflecting on the conversation, ensuring it is specific to the image.\n3. Generate a concise and relevant answer based on the conversation snippet.\n\n# Output Format\n\nThe output should be formatted as a plain-text Q&A pair:\n\n**Q:** [Question]  \n**A:** [Answer]  \n\n# Example\n\nGiven conversation snippet:  \nUser A: \"The painting of the sunset over the lake was so serene. Did you notice how the colors transitioned from orange to purple across the water?\"  \nUser B: \"Yes, and the reflection made it look like there were two sunsets. It was breathtaking.\"  \n\n**Generated Q&A Pair:**  \n**Q:** What made the lake sunset painting unique?  \n**A:** The reflection in the water made it look like there were two sunsets.")
                    QAs.append([response.splitlines()[0].replace("**Q:** ", "").strip(), response.splitlines()[1].replace("**A:** ", "").strip(), lines[0][6:], "single"])

In [22]:
# Computation
df = pd.DataFrame(QAs, columns=['question', 'answer', 'file', 'type'])

In [23]:
# Computation
combined_df = pd.concat([df, existing_QAs], ignore_index=True)

In [ ]:
# Computation
combined_df.to_csv('./augmented/query_set.csv')